Example 5 - Unsupervised: NLP Topic Discovery / Topic Modeling

### Learning pipeline
**Text acquisition → inspection → preprocessing → feature extraction → topic modeling → topic terms → document-topic probabilities → interpretation → evaluation/refinement**

Topic discovery is **unsupervised NLP**: unlike sentiment analysis, the documents do not need known topic labels.


> **Updated guided version:** This notebook retains the original topic-discovery exercise and adds an interactive LDA section where students can modify the number of topics, top words, iterations, learning method, and vocabulary thresholds, then inspect and assign topic labels.


In [ ]:
# PART 1 — IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(42)
print("Libraries loaded.")


## Part 2 — Synthetic Dataset
For teaching purposes, the corpus contains recurring themes involving teaching, facilities/internet, enrollment, student services, and safety/DRRM. **These themes are not supplied to the model as labels.**


In [ ]:
# PART 2 — CREATE SYNTHETIC TEXT DATA
comments = [
"The instructor explains programming concepts clearly and provides useful examples.",
"The laboratory exercises help us understand machine learning algorithms.",
"The lectures are informative but we need more hands-on coding activities.",
"The professor gives helpful feedback on programming assignments.",
"More practical exercises would improve our understanding of the lessons.",
"The course materials make difficult concepts easier to understand.",
"The computer laboratory needs faster computers and better internet access.",
"The campus WiFi connection is slow during laboratory classes.",
"Some computers in the laboratory are outdated and frequently freeze.",
"The internet connection disconnects during online activities.",
"The laboratory needs additional computers for students.",
"Better WiFi coverage would improve access to online learning resources.",
"The enrollment process takes too long because of long queues.",
"Online registration is convenient but the portal sometimes becomes unavailable.",
"Students need clearer instructions about enrollment requirements.",
"The registration portal should show the status of submitted documents.",
"Enrollment would be easier if more steps were available online.",
"The registration process needs faster verification of student records.",
"The library staff are helpful when students need research materials.",
"Students need more academic advising and consultation services.",
"The guidance office provides useful support for student concerns.",
"More library resources should be available for thesis students.",
"Students would benefit from longer library operating hours.",
"Academic advising helps students select appropriate subjects.",
"Emergency evacuation signs should be visible in every building.",
"Students need regular earthquake and fire evacuation drills.",
"Campus emergency announcements should provide clear safety instructions.",
"Laboratories should have accessible emergency exits and safety equipment.",
"Disaster preparedness training can help students respond during emergencies.",
"Emergency response information should be available through the campus mobile application."
]
df=pd.DataFrame({"Document":comments})
print("Documents:",len(df))
display(df.head(10))


In [ ]:
# PART 3 — INSPECT THE CORPUS
print("Shape:",df.shape)
print("\nMissing values:\n",df.isnull().sum())
print("\nDuplicates:",df.duplicated().sum())
print("\nSample documents:")
for i,text in enumerate(df["Document"].head(5),1):
    print(f"{i}. {text}")


In [ ]:
# PART 4 — BASIC TEXT PREPROCESSING
def clean_text(text):
    # Convert to lowercase.
    text=text.lower()
    # Keep letters and spaces for this introductory English example.
    text=re.sub(r"[^a-z\s]"," ",text)
    # Normalize repeated spaces.
    return re.sub(r"\s+"," ",text).strip()

df["Clean_Text"]=df["Document"].apply(clean_text)
display(df[["Document","Clean_Text"]].head())

# GUIDE:
# Do not remove words mechanically in a thesis.
# Every preprocessing decision should be justified for the language and task.


In [ ]:
# PART 5 — FEATURE EXTRACTION FOR LDA
# LDA is commonly demonstrated with word-count features.
vectorizer=CountVectorizer(stop_words="english",min_df=2)
X_counts=vectorizer.fit_transform(df["Clean_Text"])
terms=vectorizer.get_feature_names_out()

print("Document-term matrix:",X_counts.shape)
print("Vocabulary size:",len(terms))
print("Vocabulary:",terms)


In [ ]:
# PART 6 — VIEW THE DOCUMENT-TERM MATRIX
# Dense conversion is acceptable for this tiny teaching dataset.
# Keep large research matrices sparse.
dtm=pd.DataFrame(X_counts.toarray(),columns=terms)
display(dtm.head())


In [ ]:
# PART 7 — TRAIN LDA
# We initially request 5 topics for demonstration.
# In actual research, investigate and justify the number of topics.
n_topics=5
lda=LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",
    max_iter=30
)
lda.fit(X_counts)
print("LDA trained with",n_topics,"topics.")


In [ ]:
# PART 8 — DISPLAY TOP WORDS
def get_topics(model,feature_names,top_n=8):
    rows=[]
    for topic_id,weights in enumerate(model.components_):
        idx=weights.argsort()[-top_n:][::-1]
        words=[feature_names[i] for i in idx]
        rows.append({"Topic":f"Topic {topic_id}",
                     "Top Words":", ".join(words)})
    return pd.DataFrame(rows)

topic_words=get_topics(lda,terms,8)
display(topic_words)

# STUDENT TASK:
# Propose descriptive names based on the ACTUAL words produced.
# Do not force predetermined topic names onto the output.


In [ ]:
# PART 9 — DOCUMENT-TOPIC PROBABILITIES
topic_probs=lda.transform(X_counts)
prob_df=pd.DataFrame(
    topic_probs,
    columns=[f"Topic_{i}" for i in range(n_topics)]
)
display(prob_df.head(10).round(3))

# LDA permits a document to contain a mixture of topics.


In [ ]:
# PART 10 — DOMINANT TOPIC FOR EACH DOCUMENT
df["Dominant_Topic"]=topic_probs.argmax(axis=1)
df["Topic_Probability"]=topic_probs.max(axis=1)

display(df[["Document","Dominant_Topic","Topic_Probability"]]
        .sort_values(["Dominant_Topic","Topic_Probability"],
                     ascending=[True,False])
        .round({"Topic_Probability":3}))


In [ ]:
# PART 11 — READ REPRESENTATIVE DOCUMENTS
# Interpretation should use both top terms AND actual documents.
for topic_id in range(n_topics):
    print("\n"+"="*65)
    print("TOPIC",topic_id)
    print("="*65)
    subset=df[df["Dominant_Topic"]==topic_id].sort_values(
        "Topic_Probability",ascending=False)
    for _,row in subset.head(4).iterrows():
        print(f"{row['Topic_Probability']:.3f} | {row['Document']}")


In [ ]:
# PART 12 — TOPIC DISTRIBUTION
counts=df["Dominant_Topic"].value_counts().sort_index()
counts.plot(kind="bar",figsize=(8,5))
plt.xlabel("Dominant Topic")
plt.ylabel("Number of Documents")
plt.title("Distribution of Dominant Topics")
plt.xticks(rotation=0)
plt.show()
print(counts)


## Part 13 — Selecting the Number of Topics
There is no universal automatic choice for the correct number of topics. We can inspect quantitative diagnostics together with **interpretability, distinctiveness, stability, and research relevance**.

The next cell demonstrates LDA perplexity. Lower perplexity can indicate better predictive fit, but the lowest value does **not automatically mean the most meaningful topic solution**.


In [ ]:
# PART 13 — COMPARE CANDIDATE TOPIC COUNTS
rows=[]
for k in range(2,8):
    model=LatentDirichletAllocation(
        n_components=k,random_state=42,
        learning_method="batch",max_iter=30)
    model.fit(X_counts)
    rows.append([k,model.perplexity(X_counts)])

comparison=pd.DataFrame(rows,columns=["Number_of_Topics","Perplexity"])
display(comparison.round(2))

plt.figure(figsize=(8,5))
plt.plot(comparison["Number_of_Topics"],
         comparison["Perplexity"],marker="o")
plt.xlabel("Number of Topics")
plt.ylabel("Perplexity")
plt.title("LDA Topic-Number Comparison")
plt.show()


## Part 14 — Alternative Topic Discovery: TF-IDF + K-Means
Topic discovery can also be approached as **document clustering**. TF-IDF represents the documents numerically, and K-Means groups similar document vectors. This is useful for comparison but is conceptually different from probabilistic LDA.


In [ ]:
# PART 14 — TF-IDF FEATURES
tfidf=TfidfVectorizer(stop_words="english",min_df=2)
X_tfidf=tfidf.fit_transform(df["Clean_Text"])
tfidf_terms=tfidf.get_feature_names_out()
print("TF-IDF matrix:",X_tfidf.shape)


In [ ]:
# PART 15 — K-MEANS DOCUMENT CLUSTERING
k=5
km=KMeans(n_clusters=k,random_state=42,n_init=20)
labels=km.fit_predict(X_tfidf)
df["KMeans_Cluster"]=labels

print("Silhouette score:",
      round(silhouette_score(X_tfidf,labels),3))
display(df[["Document","KMeans_Cluster"]]
        .sort_values("KMeans_Cluster"))


In [ ]:
# PART 16 — TOP TERMS PER K-MEANS CLUSTER
for cluster_id in range(k):
    center=km.cluster_centers_[cluster_id]
    idx=center.argsort()[-8:][::-1]
    words=[tfidf_terms[i] for i in idx]
    print(f"Cluster {cluster_id}: {', '.join(words)}")


## Part 17 — LDA vs. TF-IDF + K-Means

| Aspect | LDA | TF-IDF + K-Means |
|---|---|---|
| Approach | Probabilistic topic modeling | Document clustering |
| Features | Commonly word counts | Commonly TF-IDF |
| Membership | Topic probability mixture | Usually one cluster |
| Main output | Topics + probabilities | Clusters + centroids |
| Interpretation | Top words + representative documents | Top terms + cluster documents |

Neither method should be selected only because it produces attractive-looking output. The choice should match the research question and data.


## Part 18 — Thesis Applications
Topic discovery can be used with:
- open-ended student feedback;
- citizen complaints;
- policy documents;
- research abstracts;
- disaster-related social-media posts;
- customer feedback;
- incident reports; and
- institutional document collections.

**Example research question:** *What recurring themes are present in open-ended student feedback regarding university services?*

Example research pipeline:

**Collect → anonymize → inspect → preprocess → extract features → discover topics → inspect representative documents → interpret/name topics → validate → report limitations and implications**


## Part 19 — Research and Ethical Guide
For real datasets, students should document privacy/anonymization, inclusion criteria, language, preprocessing, feature representation, model settings, topic-number selection, representative documents, interpretation procedure, and limitations.

For Filipino, Bikol, Taglish, or multilingual corpora, English-only preprocessing should **not** be adopted automatically. Stop-word lists, tokenization, spelling variation, code-switching, and local language meaning require appropriate treatment.

Automatically discovered topics are **not ground truth**. Topic names are researcher interpretations that require evidence from top terms and representative documents.


# Part 20 — Student Thesis Adaptation

Complete the following for your thesis:
1. Research problem and intended topic-discovery purpose.
2. Unit of analysis (comment, document, complaint, post, etc.).
3. Text source and corpus size.
4. Language(s) present.
5. Privacy/anonymization requirements.
6. Text preprocessing decisions.
7. Feature representation.
8. Candidate topic-discovery method.
9. Procedure for selecting the number of topics.
10. Procedure for naming/interpreting topics.
11. Quantitative and qualitative validation.
12. Limitations and possible biases.
13. Next experiment.

### Required Output
Submit the completed notebook and a **Topic Discovery Report** containing the dataset description, preprocessing, feature extraction, model configuration, top terms, representative documents, topic interpretations, diagnostics, limitations, and proposed improvements.


# Guided Interactive Topic Discovery

This section extends the topic-discovery exercise so that students can **change important LDA settings themselves** and observe how the discovered topics change.

### Parameters the user can enter
- **Number of topics** (`n_components`)
- **Number of top words** displayed for each topic
- **Maximum iterations** (`max_iter`)
- **Learning method** (`batch` or `online`)
- **Random state**
- **Minimum document frequency** (`min_df`)
- **Maximum document frequency** (`max_df`)

The model will then:
1. vectorize the text;
2. fit an LDA topic model;
3. display top words for every topic;
4. assign each document its dominant topic;
5. calculate topic probabilities;
6. generate a **suggested human-readable label** for each topic;
7. allow the user to **accept or replace the suggested labels**;
8. display representative documents and the final labeled results.

> **Important:** LDA discovers statistical word patterns. Topic numbers such as `Topic 0` have no inherent meaning. Labels are interpretations added after examining top words and representative documents.


In [ ]:
# INTERACTIVE SETTINGS
# GUIDE: Run this cell and enter the settings you want to experiment with.
# GUIDE: Press Enter to use the displayed default.

def ask_int(prompt, default, minimum=1):
    while True:
        raw = input(f"{prompt} [default={default}]: ").strip()
        if raw == "":
            return default
        try:
            value = int(raw)
            if value >= minimum:
                return value
            print(f"Please enter a value >= {minimum}.")
        except ValueError:
            print("Please enter a whole number.")

def ask_float(prompt, default, minimum=0.0, maximum=1.0):
    while True:
        raw = input(f"{prompt} [default={default}]: ").strip()
        if raw == "":
            return default
        try:
            value = float(raw)
            if minimum <= value <= maximum:
                return value
            print(f"Please enter a value from {minimum} to {maximum}.")
        except ValueError:
            print("Please enter a numeric value.")

n_topics = ask_int("Number of topics", 5, 2)
n_words = ask_int("Number of top words per topic", 8, 3)
max_iter = ask_int("Maximum LDA iterations", 20, 1)
random_state = ask_int("Random state", 42, 0)
min_df = ask_int("Minimum document frequency (min_df)", 1, 1)
max_df = ask_float("Maximum document frequency proportion (max_df)", 0.95, 0.01, 1.0)

while True:
    learning_method = input("Learning method: batch or online [default=batch]: ").strip().lower()
    if learning_method == "":
        learning_method = "batch"
    if learning_method in ["batch", "online"]:
        break
    print("Please type 'batch' or 'online'.")

print("\nSELECTED SETTINGS")
print("-----------------")
print("Topics:", n_topics)
print("Top words:", n_words)
print("Iterations:", max_iter)
print("Learning method:", learning_method)
print("Random state:", random_state)
print("min_df:", min_df)
print("max_df:", max_df)

# GUIDE:
# n_topics controls how many latent themes LDA attempts to discover.
# n_words affects DISPLAY only; it does not change the fitted topic model.
# max_iter controls how many passes are allowed for model optimization.
# batch uses all documents during each update; online updates incrementally.
# random_state improves reproducibility.
# min_df removes extremely rare terms.
# max_df removes terms appearing in too large a proportion of documents.


## Prepare the Text Column

The original notebook uses cleaned textual comments. The next cell automatically looks for a likely cleaned-text column. If none is found, it creates one from a likely raw-text column.

The same preprocessing should be applied consistently before topic modeling.


In [ ]:
# GUIDE: Detect the text column created in the earlier part of the notebook.

candidate_clean = ["Cleaned_Text", "cleaned_text", "Clean_Text", "clean_text", "Cleaned Comment"]
candidate_raw = ["Comment", "comment", "Text", "text", "Feedback", "feedback"]

text_col = next((c for c in candidate_clean if c in df.columns), None)

if text_col is None:
    raw_col = next((c for c in candidate_raw if c in df.columns), None)
    if raw_col is None:
        raise ValueError(
            "No recognizable text column was found. "
            "Please set text_col manually to the column containing the cleaned comments."
        )
    text_col = raw_col

print("Text column used for topic discovery:", text_col)
print("Number of documents:", len(df))
display(df[[text_col]].head())


## Vectorization

LDA works with a **document-term count matrix**. `CountVectorizer` converts the collection of comments into numerical word counts.

`min_df` and `max_df` can change the vocabulary substantially, so students should record these values when reporting experiments.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# GUIDE: stop_words="english" removes common English function words.
vectorizer_interactive = CountVectorizer(
    stop_words="english",
    min_df=min_df,
    max_df=max_df
)

X_counts_interactive = vectorizer_interactive.fit_transform(
    df[text_col].astype(str)
)

feature_names_interactive = np.array(
    vectorizer_interactive.get_feature_names_out()
)

print("Document-term matrix shape:", X_counts_interactive.shape)
print("Documents:", X_counts_interactive.shape[0])
print("Vocabulary size:", X_counts_interactive.shape[1])

# GUIDE:
# Rows = documents/comments.
# Columns = vocabulary terms.
# Values = word counts.


## Fit the LDA Model

The user-selected hyperparameters are now applied.

The model returns a probability distribution over topics for each document. A document can contain more than one topic, but its **dominant topic** is the one with the highest probability.


In [ ]:
lda_interactive = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=max_iter,
    learning_method=learning_method,
    random_state=random_state
)

document_topic = lda_interactive.fit_transform(X_counts_interactive)

print("LDA training completed.")
print("Number of discovered topics:", lda_interactive.n_components)
print("Iterations actually performed:", lda_interactive.n_iter_)
print("Perplexity:", round(lda_interactive.perplexity(X_counts_interactive), 3))

# GUIDE:
# Lower perplexity can indicate better statistical fit when comparing
# models fitted to the same dataset and representation.
# Do NOT choose a topic model using perplexity alone.
# Interpretability and usefulness of the topics also matter.


## Display Top Words and Generate Suggested Labels

Each LDA topic is represented by words with different weights. The highest-weighted words provide clues about the meaning of the topic.

The notebook generates a **suggested label** by combining prominent topic words. This is only a starting point—not an authoritative semantic label. Students should inspect the top words and representative documents before naming a topic.


In [ ]:
topic_word_lists = {}
suggested_labels = {}

print("TOP WORDS PER TOPIC")
print("===================")

for topic_idx, topic_weights in enumerate(lda_interactive.components_):
    top_indices = topic_weights.argsort()[-n_words:][::-1]
    top_words = feature_names_interactive[top_indices].tolist()
    topic_word_lists[topic_idx] = top_words

    # Simple automatic label suggestion using the first 3 prominent words.
    # GUIDE: This is intentionally transparent and easy to explain.
    label_words = top_words[:3]
    suggested = " / ".join(w.title() for w in label_words)
    suggested_labels[topic_idx] = suggested

    print(f"\nTopic {topic_idx}")
    print("Top words:", ", ".join(top_words))
    print("Suggested label:", suggested)


## Dominant Topic and Representative Documents

Before finalizing labels, inspect actual documents strongly associated with each topic. This is essential because top words alone can sometimes be ambiguous.


In [ ]:
results_topic = df.copy()

results_topic["Dominant_Topic"] = document_topic.argmax(axis=1)
results_topic["Topic_Probability"] = document_topic.max(axis=1)

for k in range(n_topics):
    results_topic[f"Topic_{k}_Probability"] = document_topic[:, k]

for k in range(n_topics):
    print("\n" + "=" * 75)
    print(f"TOPIC {k}: {suggested_labels[k]}")
    print("Top words:", ", ".join(topic_word_lists[k]))
    print("-" * 75)

    reps = (
        results_topic[results_topic["Dominant_Topic"] == k]
        .sort_values("Topic_Probability", ascending=False)
        [[text_col, "Topic_Probability"]]
        .head(5)
    )
    display(reps)

# GUIDE:
# Representative documents have high probability for the selected topic.
# Read them together with the top words before deciding the final label.


## User-Defined Topic Labels

Now you can assign meaningful labels.

For each topic, the notebook shows its top words and suggested label. Press **Enter** to accept the suggestion, or type your own label such as:

- `Teaching and Learning`
- `Facilities and Internet`
- `Enrollment and Registration`
- `Student Services`
- `Safety and Disaster Preparedness`

The exact labels should be based on the topics actually discovered in the current run.


In [ ]:
final_topic_labels = {}

print("ASSIGN FINAL TOPIC LABELS")
print("=========================")

for k in range(n_topics):
    print(f"\nTopic {k}")
    print("Top words:", ", ".join(topic_word_lists[k]))
    print("Suggested label:", suggested_labels[k])

    user_label = input(
        "Enter your preferred label, or press Enter to accept suggestion: "
    ).strip()

    if user_label == "":
        user_label = suggested_labels[k]

    final_topic_labels[k] = user_label

print("\nFINAL LABELS")
for k, label in final_topic_labels.items():
    print(f"Topic {k}: {label}")


## Apply the Final Labels

The topic number is retained for reproducibility, while the human-readable label makes the output easier to interpret in a thesis, report, or application.


In [ ]:
results_topic["Topic_Label"] = results_topic["Dominant_Topic"].map(final_topic_labels)

output_columns = [
    text_col,
    "Dominant_Topic",
    "Topic_Label",
    "Topic_Probability"
]

print("LABELED TOPIC-DISCOVERY RESULTS")
display(
    results_topic[output_columns]
    .sort_values(["Dominant_Topic", "Topic_Probability"], ascending=[True, False])
)

print("\nTOPIC DISTRIBUTION")
topic_summary = (
    results_topic.groupby(["Dominant_Topic", "Topic_Label"])
    .size()
    .reset_index(name="Documents")
)
topic_summary["Percentage"] = (
    topic_summary["Documents"] / len(results_topic) * 100
).round(2)

display(topic_summary)


## Visualize Topic Distribution

This graph shows how many documents were assigned to each dominant topic. Remember that LDA is probabilistic: a document can contain several topics even though the table reports only its strongest topic as the dominant topic.


In [ ]:
plot_summary = topic_summary.copy()
plot_summary["Display_Label"] = (
    "Topic " + plot_summary["Dominant_Topic"].astype(str)
    + ": " + plot_summary["Topic_Label"]
)

plt.figure(figsize=(10, 5))
plt.bar(plot_summary["Display_Label"], plot_summary["Documents"])
plt.xlabel("Topic")
plt.ylabel("Number of Documents")
plt.title("Distribution of Dominant Topics")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## Inspect the Probability Distribution of a Document

The next cell allows the student to enter a document row number and see its probability across all discovered topics. This demonstrates that LDA does not simply make a hard classification.


In [ ]:
while True:
    raw = input(f"Enter a document row number from 0 to {len(results_topic)-1}: ").strip()
    try:
        doc_index = int(raw)
        if 0 <= doc_index < len(results_topic):
            break
        print("Row number is outside the valid range.")
    except ValueError:
        print("Please enter a whole number.")

print("\nDOCUMENT")
print(results_topic.iloc[doc_index][text_col])

prob_table = pd.DataFrame({
    "Topic": [f"Topic {k}" for k in range(n_topics)],
    "Label": [final_topic_labels[k] for k in range(n_topics)],
    "Probability": document_topic[doc_index]
}).sort_values("Probability", ascending=False)

display(prob_table)

print(
    "Dominant topic:",
    results_topic.iloc[doc_index]["Dominant_Topic"],
    "-",
    results_topic.iloc[doc_index]["Topic_Label"]
)


## Optional Experiment: Compare Different Numbers of Topics

There is no universal “correct” number of topics. Students can rerun the interactive settings with different values—for example 3, 4, 5, or 6—and compare:

- topic interpretability;
- overlap among top words;
- representative documents;
- topic sizes;
- perplexity;
- usefulness for the research objective.

Changing the number of topics can also change the meaning of the discovered topics. Therefore, **labels should be reviewed again after every new model run**.


## Student Interpretation Guide

After running the interactive topic model, answer:

1. How many documents were analyzed?
2. How many topics did you request?
3. What values did you use for `max_iter`, `min_df`, and `max_df`?
4. What were the top words for each topic?
5. What label did you assign to each topic, and why?
6. Which representative documents support each label?
7. Which topic contained the most documents?
8. Were there documents with low dominant-topic probabilities? What might this mean?
9. What happened when you changed the number of topics?
10. Did increasing the iterations noticeably change the topics?
11. Were any topics difficult to distinguish from one another?
12. Which topic configuration is most interpretable for the research problem?

### Thesis reminder

Report the preprocessing procedure, vectorization settings, LDA hyperparameters, topic-selection rationale, top words, representative documents, labeling procedure, and limitations. Topic labels are **researcher interpretations of statistical patterns**, so the basis for each label should be documented.
